# 🧹 Task 2: Data Cleaning & Missing Value Handling
## Edutech Solution – AI & ML Internship

**Dataset:** Titanic (891 rows × 12 columns)  
**Tools:** Python · Pandas · NumPy · Matplotlib · Seaborn · Scikit-learn  
**Objective:** Master the art of data preprocessing and quality assurance

---

## 📦 Step 0 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings("ignore")

print("✅ All libraries imported successfully!")

## 📂 Step 1 — Load Dataset

In [ ]:
df = pd.read_csv("titanic.csv")
print(f"Dataset Shape: {df.shape[0]} rows × {df.shape[1]} columns")
df.head()

## ❓ Step 2 — Identify Missing Values
> `isnull().sum()` counts missing entries per column.

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({"Missing Count": missing, "Missing (%)": missing_pct, "Dtype": df.dtypes})
print("Missing Value Summary:")
missing_df[missing_df["Missing Count"] > 0]

## 🗺️ Step 3 — Visualize Missing Data (Heatmap)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Heatmap
sns.heatmap(df.isnull(), yticklabels=False, cbar=True, cmap='viridis', ax=axes[0])
axes[0].set_title("Missing Value Heatmap (Yellow = Missing)", fontsize=13, fontweight='bold')

# Bar chart
missing_cols = df.columns[df.isnull().any()]
(df[missing_cols].isnull().sum() / len(df) * 100).plot(kind='bar', color=['#e74c3c','#e67e22','#9b59b6'], 
                                                         ax=axes[1], edgecolor='black')
axes[1].set_title("Missing % per Column", fontsize=13, fontweight='bold')
axes[1].set_ylabel("Missing (%)")
axes[1].set_xticklabels(missing_cols, rotation=15)

plt.tight_layout()
plt.savefig("missing_value_heatmap.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Heatmap saved!")

## 📖 Step 4 — Types of Missing Data
| Column | Type | Reason |
|--------|------|--------|
| Age | **MAR** (Missing At Random) | Correlates with Pclass/Sex |
| Cabin | **MNAR** (Missing Not At Random) | Lower-class passengers had no cabin |
| Embarked | **MCAR** (Missing Completely At Random) | Only 2 rows, random |

## 🔧 Step 5 — Handle Missing Values
### A) Drop 'Cabin' — 77% missing

In [ ]:
df_clean = df.copy()

# Drop Cabin — too many missing values
df_clean.drop(columns=["Cabin"], inplace=True)
print(f"After dropping Cabin: {df_clean.shape[1]} columns")
df_clean.isnull().sum()

### B) Median Imputation — Age
> Use **Median** (not Mean) because Age has outliers — median is robust.

In [ ]:
age_median = df_clean["Age"].median()
print(f"Median Age = {age_median}")

df_clean["Age"] = df_clean["Age"].fillna(age_median)
print(f"Missing Age after imputation: {df_clean['Age'].isnull().sum()}")

### C) Mode Imputation — Embarked
> Use **Mode** for categorical columns.

In [ ]:
embarked_mode = df_clean["Embarked"].mode()[0]
print(f"Mode Embarked = '{embarked_mode}'")

df_clean["Embarked"] = df_clean["Embarked"].fillna(embarked_mode)
print(f"Missing Embarked after imputation: {df_clean['Embarked'].isnull().sum()}")

## 🤖 Step 6 — SimpleImputer (Scikit-learn)
> Automated, pipeline-friendly imputation using Scikit-learn.

In [ ]:
df_si = df[["Age", "Fare"]].copy()
print(f"Before: Age={df_si['Age'].isnull().sum()} missing, Fare={df_si['Fare'].isnull().sum()} missing")

imputer = SimpleImputer(strategy="median")
df_si[["Age", "Fare"]] = imputer.fit_transform(df_si[["Age", "Fare"]])

print(f"After : Age={df_si['Age'].isnull().sum()} missing, Fare={df_si['Fare'].isnull().sum()} missing")
print(f"Imputed values → Age: {imputer.statistics_[0]:.1f}, Fare: {imputer.statistics_[1]:.2f}")

## 📦 Step 7 — Outlier Detection & Treatment (IQR Method)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Outlier Detection – Boxplots Before Winsorization", fontsize=14, fontweight='bold')

outlier_cols = ["Age", "Fare", "SibSp", "Parch"]

for i, col in enumerate(outlier_cols):
    ax = axes[i // 2][i % 2]
    Q1, Q3 = df_clean[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    n_outliers = df_clean[(df_clean[col] < lower) | (df_clean[col] > upper)].shape[0]
    
    ax.boxplot(df_clean[col].dropna(), patch_artist=True,
               boxprops=dict(facecolor='#3498db', alpha=0.7))
    ax.set_title(f"'{col}' — {n_outliers} outliers (IQR method)", fontsize=11)
    
    # Cap outliers
    df_clean[col] = np.clip(df_clean[col], lower, upper)

plt.tight_layout()
plt.savefig("outlier_boxplots.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Outliers capped using Winsorization!")

## 🧹 Step 8 — Additional Cleaning

In [ ]:
# Drop non-informative columns
df_clean.drop(columns=["PassengerId", "Name", "Ticket"], inplace=True)

# Encode categorical
df_clean["Sex"]      = df_clean["Sex"].map({"male": 0, "female": 1})
df_clean["Embarked"] = df_clean["Embarked"].map({"S": 0, "C": 1, "Q": 2})

# Remove duplicates
df_clean.drop_duplicates(inplace=True)
df_clean.reset_index(drop=True, inplace=True)

print(f"✅ Final shape: {df_clean.shape}")
df_clean.head()

## ✅ Step 9 — Verify Clean Dataset

In [ ]:
print("🔍 Final Verification:")
print(f"   Missing values  : {df_clean.isnull().sum().sum()}")
print(f"   Duplicate rows  : {df_clean.duplicated().sum()}")
print(f"   Final shape     : {df_clean.shape}")
print()
df_clean.info()

## 📊 Step 10 — Before vs After Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df["Age"].dropna().hist(bins=30, color='#e74c3c', alpha=0.7, ax=axes[0], edgecolor='black')
axes[0].set_title("Age — Before Cleaning", fontsize=12)
df_clean["Age"].hist(bins=30, color='#2ecc71', alpha=0.7, ax=axes[1], edgecolor='black')
axes[1].set_title("Age — After Cleaning", fontsize=12)
plt.tight_layout()
plt.savefig("age_before_after.png", dpi=150, bbox_inches='tight')
plt.show()

## 💾 Save Cleaned Dataset

In [ ]:
df_clean.to_csv("titanic_cleaned.csv", index=False)
print(f"✅ titanic_cleaned.csv saved — {df_clean.shape[0]} rows × {df_clean.shape[1]} columns")
df_clean.describe().round(2)

## 💡 Interview Answers

**Q1. Types of missing data (MCAR, MAR, MNAR)?**  
- **MCAR** – Missing Completely At Random (e.g., Embarked — 2 random entries)  
- **MAR** – Missing At Random (e.g., Age — relates to passenger class)  
- **MNAR** – Missing Not At Random (e.g., Cabin — lower-class had none)  

**Q2. Drop rows vs imputing values?**  
Drop when >40–50% data is missing or the column is irrelevant. Impute when data is MAR/MCAR and the feature is important.

**Q3. How do outliers affect imputation?**  
Outliers inflate the Mean, making mean-imputation biased. Median is more robust and preferred when outliers are present.

**Q4. Mean vs Median imputation?**  
Mean is good for normally distributed data. Median is better when the distribution is skewed or contains outliers.

**Q5. Role of data cleaning in the ML pipeline?**  
Garbage-in = garbage-out. Cleaning ensures the model trains on accurate, consistent data — improving accuracy, reducing bias, and preventing errors.